# 09 · Numba CUDA ②: 히스토그램 (atomic · 공유메모리)

> **CuPy 2일 집중 코스 — Day 2 / 단원 6 (Numba CUDA 커널 작성, GTC 07 기반)**

히스토그램으로 **데이터 레이스 → atomic → 공유메모리**의 최적화 단계를 직접 구현합니다.
07에서 배운 **동기화/atomic(6절)·공유메모리(4절)** 개념이 실제로 어떻게 쓰이는지 봅니다.

### 왜 하필 히스토그램인가

히스토그램(각 값의 등장 횟수 세기)은 겉보기엔 사소하지만, **병렬 프로그래밍에서 가장 흔히
마주치는 "여러 스레드가 같은 메모리 위치를 동시에 갱신"하는 패턴의 가장 단순한 대표 사례**입니다.
원소별 연산(`08`의 copy/scale)은 각 스레드가 서로 다른 주소만 건드리므로 충돌이 없지만, 히스토그램은
입력값에 따라 **여러 스레드가 같은 칸(bin)을 두고 경쟁**합니다. 이 충돌을 어떻게 다루느냐가 곧
리덕션(reduce), 정렬(counting sort), 그래프 알고리즘의 degree 계산 등 수많은 병렬 알고리즘의 핵심
난제와 직결됩니다. 그래서 GTC 07 강의도 이 예제로 atomic과 공유메모리를 함께 가르칩니다.

### 이 노트북의 흐름

naive 구현으로 **데이터 레이스를 눈으로 확인**(1~2절) → `cuda.atomic.add`로 **정확성 복구**(3절) →
공유메모리 privatization으로 **경합 자체를 줄여 가속**(4절) → 두 버전 성능 비교(5절) → (선택)
cooperative load·Nsight Compute(6절). `08`이 "같은 답을 내는 두 커널의 속도 차이"(coalescing)를
다뤘다면, 09는 "**틀린 답에서 출발해 맞는 답으로, 그다음 빠른 답으로**" 나아가는 조금 다른 종류의
최적화 여정입니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **6. 동기화/atomic**: 데이터 레이스 → `cuda.atomic.add` → `cuda.syncthreads`
- **4. 메모리 계층**: 블록별 **공유 메모리(`cuda.shared.array`)** 로 전역 경합 감소

## 학습 목표
- 병렬 누적의 **데이터 레이스**를 이해하고 `atomic`으로 고친다.
- **공유메모리 privatization**으로 전역 atomic 경합을 줄여 가속한다.
- 왜 "정확성 먼저, 성능은 그다음"이 병렬 코드 디버깅의 기본 순서인지 체득한다.

## 목차
1. [히스토그램 & 전역 naive(레이스)](#1)
2. [데이터 레이스 진단](#2)
3. [atomic으로 수정](#3)
4. [공유메모리 최적화](#4)
5. [성능 비교](#5)
6. [(선택) cooperative & ncu](#6)
7. [체크포인트](#7)

> 필요: `numba`. 데이터는 자체 생성(난수 바이트). GTC 원본은 책 텍스트(문자 빈도)를 사용합니다.

In [1]:
import numpy as np, cupy as cp, numba
from numba import cuda
from course_utils import print_env, bench, gpu_ms, print_bench
print_env()

BINS = 256
N = 1 << 24
values = cp.random.randint(0, BINS, size=N, dtype=cp.int32)  # 0..255 값
ref = np.bincount(cp.asnumpy(values), minlength=BINS)        # 정답(CPU)

=== Environment ===
numpy: 2.3.5
cupy : 14.1.1
device: 0 - NVIDIA A100-SXM4-80GB (compute capability 8.0)
VRAM  : 79.25 GB


<a id="1"></a>
## 1. 히스토그램 & 전역 naive(레이스)

히스토그램 = 각 값의 등장 횟수 세기. 가장 단순한 구현: 각 스레드가 자기 값의 칸을 +1.
하지만 `hist[v] = hist[v] + 1` 은 **읽기-수정-쓰기**라 여러 스레드가 같은 칸을 동시에 건드리면 깨집니다.

### 조금 더 구체적으로: 이 예제에서 충돌이 "가끔"이 아니라 "항상" 일어나는 이유

아래 코드 셀의 설정은 `BINS = 256`, `N = 1 << 24`(약 1,678만 개 값)입니다. 값이 0~255 사이에 **균등
분포**하도록 만들었으므로, 칸(bin) 하나에는 평균적으로 `N / BINS ≈ 65,536`개의 값이 몰립니다. 즉
같은 주소(`hist[v]`)를 놓고 **평균 6만 5천 개의 스레드가 경쟁**한다는 뜻입니다. 게다가 `threads=256`
블록을 수만 개 런치하면, 물리적으로 GPU의 수천 개 코어가 **정확히 같은 순간에** 서로 다른 워프에서
같은 주소에 접근하는 일이 실제로 매 마이크로초 벌어집니다. 08의 copy 커널처럼 스레드마다 전용
주소만 건드리는 경우와 근본적으로 다른 지점이 바로 여기입니다 — **08은 경합이 아예 없고, 09는
경합이 설계상 필연적**입니다.

`hist[v] = hist[v] + 1`이 컴파일되면 하드웨어 수준에서 대략 세 단계로 쪼개집니다.
1. **LOAD**: `hist[v]`의 현재 값을 레지스터로 읽음
2. **ADD**: 레지스터 값에 1을 더함
3. **STORE**: 결과를 다시 `hist[v]`에 씀

이 세 단계는 **하나의 원자적 단위가 아니라 서로 독립적인 세 개의 GPU 명령**입니다. 그래서 다른
스레드가 이 세 단계 "사이"에 끼어들 여지가 생깁니다 — 정확히 이 지점이 다음 절(2)에서 다루는
데이터 레이스입니다.

In [10]:
@cuda.jit
def hist_naive(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        v = values[i]
        hist[v] = hist[v] + 1      # ⚠️ 데이터 레이스

threads = 256; blocks = (N + threads - 1)//threads
hist = cp.zeros(BINS, dtype=cp.int32)
hist_naive[blocks, threads](values, hist)
cp.cuda.Device().synchronize()
print('naive 합계:', int(hist.sum()), '/ 기대:', N)   # 합계가 모자람!

naive 합계: 42087 / 기대: 16777216


In [15]:
@cuda.jit
def hist_naive(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        v = values[i]
        hist[v] += 1      # ⚠️ 데이터 레이스

threads = 256; blocks = (N + threads - 1)//threads
hist = cp.zeros(BINS, dtype=cp.int32)
hist_naive[blocks, threads](values, hist)
cp.cuda.Device().synchronize()
print('naive 합계:', int(hist.sum()), '/ 기대:', N)   # 합계가 모자람!

naive 합계: 39652 / 기대: 16777216


<a id="2"></a>
## 2. 데이터 레이스 진단

두 스레드가 같은 칸을 동시에 갱신하면:
1) T0이 count(0) 읽음 → 2) T1도 count(0) 읽음 → 3) 둘 다 1로 써서 **한 증가가 사라짐**.
그래서 위 합계가 N보다 작습니다. 해결: **나눌 수 없는 atomic 연산**.

### 조금 더 구체적으로: "사라진 갱신(lost update)"의 타임라인

앞 절의 LOAD→ADD→STORE 세 단계를 두 스레드(T0, T1)가 `hist[7]`을 동시에 갱신하는 상황에 대입하면:

| 시각 | T0 | T1 | `hist[7]`의 실제 값 |
|------|----|----|----|
| t0 | LOAD hist[7] → 0 | | 0 |
| t1 | | LOAD hist[7] → 0 (T0의 갱신을 못 봄!) | 0 |
| t2 | ADD 0+1=1 | | 0 |
| t3 | STORE hist[7]=1 | | 1 |
| t4 | | ADD 0+1=1 | 1 |
| t5 | | STORE hist[7]=1 | **1** (2여야 하는데 1) |

두 번의 +1이 있었는데 결과는 한 번만 반영됐습니다 — 이를 **lost update(사라진 갱신)** 문제라
부릅니다. CPU에서는 GIL이나 락(lock)이 이런 경합을 막아주는 경우가 많지만, GPU 커널에는 그런
안전장치가 **기본으로 없습니다**. 게다가 이 문제는 **비결정적(non-deterministic)** 입니다 — 워프
스케줄링 순서는 실행마다 달라질 수 있어서, 같은 코드를 다시 실행해도 누락되는 개수가 매번 다를 수
있고 운이 나쁘면 우연히 맞는 값이 나올 수도 있습니다. 이런 특성 때문에 데이터 레이스 버그는
**테스트를 통과했다고 안심할 수 없는**, 병렬 프로그래밍에서 가장 악명 높은 버그 유형 중
하나입니다.

해결책은 LOAD-ADD-STORE 세 단계를 **하드웨어가 보장하는 하나의 원자적(atomic) 연산**으로
묶는 것입니다 — 다음 절에서 `cuda.atomic.add`로 구현합니다.

<a id="3"></a>
## 3. atomic으로 수정 — 연습

`cuda.atomic.add(array, index, value)` 는 `array[index] += value` 를 **원자적**으로 수행합니다(6절).

### 조금 더 구체적으로: atomic은 어떻게 "원자적"을 보장하나

`cuda.atomic.add`는 GPU 메모리 컨트롤러(또는 L2 캐시 단의 atomic 유닛) 수준에서 LOAD-ADD-STORE를
**하나의 되돌릴 수 없는 하드웨어 트랜잭션**으로 실행합니다. 같은 주소에 여러 스레드가 동시에
`atomic.add`를 걸면, 하드웨어가 이들을 **한 번에 하나씩 순차적으로(serialize)** 처리하도록 줄을
세웁니다 — 그래서 결과는 항상 정확하지만(2절의 lost update가 사라짐), **경합이 심한 주소일수록
그 줄이 길어져 처리량이 떨어집니다.** 즉 atomic은 "정확성"은 무조건 보장하지만 "속도"는 별개
문제입니다. 앞 절 계산대로 이 노트북의 설정에서는 칸 하나에 평균 65,536개의 스레드가 몰리므로,
**전역 메모리(global memory)의 같은 256개 주소를 놓고 GPU 전체가 경쟁**하는 셈이 되어 상당한
직렬화 병목이 생깁니다. 이 병목을 줄이는 것이 4절의 공유메모리 최적화입니다.

> 💡 `cuda.atomic.add`는 `add` 외에도 `max`, `min`, `exch`, `cas`(compare-and-swap) 등을 지원합니다
> (자세한 목록은 Numba CUDA 문서 참고). 히스토그램처럼 "더하기만" 필요한 경우가 가장 흔한
> 사용처입니다.

In [ ]:
@cuda.jit
def hist_atomic(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        # TODO: cuda.atomic.add(hist, values[i], 1) 로 안전하게 증가
        pass

hist = cp.zeros(BINS, dtype=cp.int32)
# hist_atomic[blocks, threads](values, hist); cp.cuda.Device().synchronize()
# assert int(hist.sum())==N; np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('atomic OK')

In [17]:
@cuda.jit
def hist_atomic(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        cuda.atomic.add(hist, values[i], 1)

hist = cp.zeros(BINS, dtype=cp.int32)
hist_atomic[blocks, threads](values, hist); cp.cuda.Device().synchronize()
assert int(hist.sum()) == N
np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('atomic OK')

atomic OK


<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def hist_atomic(values, hist):
    i = cuda.grid(1)
    if i < values.size:
        cuda.atomic.add(hist, values[i], 1)

hist = cp.zeros(BINS, dtype=cp.int32)
hist_atomic[blocks, threads](values, hist); cp.cuda.Device().synchronize()
assert int(hist.sum()) == N
np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('atomic OK')
```

포인트: naive 버전과 코드 한 줄 차이(`hist[v] = hist[v]+1` → `cuda.atomic.add(hist, v, 1)`)뿐이지만,
이 한 줄이 "읽기-수정-쓰기 3단계"를 "하드웨어가 보장하는 원자적 1단계"로 바꿔 데이터 레이스를
근본적으로 없앱니다. `assert int(hist.sum()) == N`이 통과하는 것이 바로 "누락된 갱신이 더 이상
없다"는 증거입니다. 다만 앞 절에서 짚었듯 정확성을 얻은 대신 **전역 주소 경합에 따른 직렬화
비용**은 그대로 남아 있습니다 — 5절의 성능 비교에서 이 비용을 수치로 확인합니다.
</details>

<a id="4"></a>
## 4. 공유메모리 최적화

전역 메모리 atomic은 **모든 블록이 같은 256칸을 두고 경합**해 느립니다.
각 블록이 **공유메모리에 자기만의 히스토그램**(privatization)을 만들어 거기서 atomic을 한 뒤,
마지막에 블록 결과만 전역에 합칩니다. 공유메모리는 온칩이라 빠르고 경합 범위가 블록 내로 줄어듭니다(4절).

### 조금 더 구체적으로: privatization이 왜 효과적인가, 숫자로 보기

앞 절의 atomic 버전은 **전역 메모리의 같은 256개 주소**를 놓고 전체 스레드(이 노트북 설정에서
`blocks = N/threads = 65,536`개 블록, 스레드 총합 1,678만 개)가 경쟁합니다. 즉 **1,678만 번의
atomic 연산이 전부 전역 메모리의 256칸을 두고 직렬화 경쟁**을 벌입니다.

공유메모리 버전(아래 `hist_shared` 커널)은 다릅니다.
1. 각 블록이 자신만의 **256칸짜리 로컬 히스토그램**(`cuda.shared.array(256, numba.int32)`, 블록당
   1 KB)을 만들고 0으로 초기화합니다. 이 초기화가 끝날 때까지 블록 내 모든 스레드를 기다리게 하는
   것이 첫 번째 `cuda.syncthreads()`입니다 — 누군가 아직 0으로 안 지운 칸을 다른 스레드가 먼저
   읽어버리는 사고를 막습니다.
2. 블록 내 스레드들이 **grid-stride 루프**로 담당 구간의 값을 읽어 **공유메모리(smem)에
   atomic.add** 합니다. 경합은 여전히 있지만 이제 그 범위가 **한 블록 안(최대 threads=256개
   스레드)** 으로 좁혀지고, 공유메모리는 온칩이라 전역 메모리보다 훨씬 빠르게 이 경합을 처리합니다.
3. 모든 스레드가 자기 몫을 다 셀 때까지 두 번째 `cuda.syncthreads()`로 기다린 뒤, 마지막으로
   **블록의 256칸 결과만** 전역 히스토그램에 `atomic.add`로 합칩니다(reduction 단계).

이 노트북은 공유메모리 버전을 `hist_shared[1024, threads]`, 즉 **1,024개 블록**으로만 런치합니다
(grid-stride 루프 덕분에 블록 수가 적어도 전체 데이터를 다 훑을 수 있습니다 — `08`의 coalesced
copy와 같은 패턴입니다). 그 결과 **전역 메모리에 대한 atomic 연산 횟수가 `1,024 × 256 =
262,144`번**으로 줄어듭니다 — atomic 버전의 1,678만 번과 비교하면 **약 64배 감소**입니다. 실제
값 누적(로컬 atomic)은 여전히 1,678만 번 일어나지만, 그 경합은 전역 256칸이 아니라 **블록마다
독립적인 로컬 256칸**에서 일어나므로 서로 다른 블록끼리는 전혀 부딪히지 않습니다. 이것이 바로
"**전역 경합을 블록 단위로 쪼갠다**"는 privatization의 핵심이며, 5절에서 이 구조가 실제로 얼마나
빨라지는지 측정합니다.

> ⚠️ `syncthreads()`가 두 번 필요한 이유를 헷갈리지 마세요: 첫 번째는 "초기화 완료 보장"(쓰기
> 시작 전), 두 번째는 "집계 완료 보장"(읽어서 전역에 합치기 전)입니다. 둘 중 하나라도 빠지면
> 또 다른 형태의 데이터 레이스가 생깁니다.

In [19]:
@cuda.jit
def hist_shared(values, hist):
    smem = cuda.shared.array(256, numba.int32)   # 블록별 히스토그램
    t = cuda.threadIdx.x; nt = cuda.blockDim.x
    j = t
    while j < 256:                # 공유메모리 0으로 초기화
        smem[j] = 0; j += nt
    cuda.syncthreads()            # 초기화 완료까지 대기
    i = cuda.grid(1); stride = cuda.gridsize(1)
    while i < values.size:        # 블록 내 atomic (경합 ↓)
        cuda.atomic.add(smem, values[i], 1); i += stride
    cuda.syncthreads()            # 집계 완료까지 대기
    j = t
    while j < 256:                # 블록 결과를 전역에 합치기
        cuda.atomic.add(hist, j, smem[j]); j += nt

hist = cp.zeros(BINS, dtype=cp.int32)
hist_shared[1024, threads](values, hist); cp.cuda.Device().synchronize()
np.testing.assert_array_equal(cp.asnumpy(hist), ref); print('shared OK')

shared OK


<a id="5"></a>
## 5. 성능 비교

전역 atomic vs 공유메모리 atomic 을 비교합니다. 값 분포가 좁을수록(경합 심할수록) 공유메모리 이득이 큽니다.

### 조금 더 구체적으로

여기서 "값 분포가 좁다"는 것은 **칸(bin) 수(`BINS`)가 적거나, 값이 몇몇 칸에 몰리는 경우**를
뜻합니다. 예를 들어 `BINS`를 256에서 16으로 줄이면 칸당 평균 값 개수가 65,536개에서 약
1,048,576개로 늘어 전역 atomic의 경합이 훨씬 심해지고, 공유메모리 버전의 상대적 이득도 커집니다.
반대로 `BINS`가 매우 커서(예: 2^20) 칸당 값이 거의 1개씩만 떨어진다면 애초에 경합이 드물어 두
버전의 차이는 작아집니다 — 즉 **공유메모리 privatization의 이득은 "경합의 심각도"에 비례**합니다.

실전 예시로는 텍스트의 문자 빈도(원래 GTC 07 예제), 이미지의 픽셀 값 분포(0~255), 로그의 HTTP
상태 코드 집계처럼 **값이 소수의 인기 있는 칸에 쏠리는(Zipf 분포에 가까운) 경우**가 흔한데, 이런
경우일수록 전역 atomic만으로는 성능이 크게 떨어지고 공유메모리 privatization의 효과가
두드러집니다. 실제 배수는 GPU 아키텍처(SM 개수, L2 캐시 크기)와 `BINS`/`N` 비율에 따라
달라지므로, 아래 측정값을 자신의 GPU에서 직접 확인해보는 것이 중요합니다.

In [ ]:
def run_atomic():
    hist[:] = 0; hist_atomic[blocks, threads](values, hist)
def run_shared():
    hist[:] = 0; hist_shared[1024, threads](values, hist)
print_bench(bench(run_atomic, n_repeat=20, n_warmup=5, name='global atomic'))
print_bench(bench(run_shared, n_repeat=20, n_warmup=5, name='shared atomic'))
print('speedup:', round(gpu_ms(bench(run_atomic))/gpu_ms(bench(run_shared)), 2))

<a id="6"></a>
## 6. (선택) cooperative load & Nsight Compute

지금까지의 공유메모리 버전도 이미 상당히 빠르지만, 값을 읽어오는 과정(load)과 히스토그램 갱신
(atomic.add)을 완전히 분리하면 조금 더 짜낼 여지가 있습니다. 아래는 그 실험적 접근과, `08`에서
이미 다룬 Nsight Compute 프로파일링을 이 커널에도 적용하는 방법입니다. (선택 사항이라 실습
시간이 부족하면 건너뛰어도 됩니다.)

<details><summary>펼쳐 보기 — 더 빠른 로드와 프로파일 </summary>

**cooperative groups**(`cuda.cooperative`)의 `coop.block.load`로 값을 **striped(coalesced)** 로 한 번에 읽어
로드와 갱신을 분리하면 더 빨라집니다. (실험적 API — 버전에 따라 인터페이스가 다를 수 있음)

```python
import cuda.cooperative.experimental as coop
block_load = coop.block.load(numba.int32, threads_per_block, items_per_thread, 'striped')
@cuda.jit(link=block_load.files)
def hist_coop(values, hist):
    items = cuda.local.array(items_per_thread, numba.int32)
    block_load(values, items)        # 연속 로드
    ...  # 공유메모리 히스토그램 갱신
```

지금 `hist_shared` 커널은 로드(`values[i]` 읽기)와 갱신(`cuda.atomic.add(smem, ...)`)이 한 줄에
섞여 있습니다. `coop.block.load`로 이 둘을 분리하면, 블록의 스레드들이 **striped 패턴**(08에서 본
coalesced 접근과 동일한 원리)으로 값을 한꺼번에 `cuda.local.array`(스레드별 레지스터에 가까운
저장소)로 읽어들인 뒤, 그다음에 각자 로컬 배열을 순회하며 공유메모리를 갱신합니다. 메모리 로드와
atomic 갱신이라는 서로 다른 두 작업의 병목을 분리해 스케줄링 유연성을 높이는 기법으로, `10_cccl`
에서 다룰 "검증된 라이브러리가 내부적으로 이미 이런 최적화를 적용해둔다"는 사실과도 연결됩니다.

**Nsight Compute**: 단계별로 `ncu`를 돌려 **Memory Workload** 처리량을 비교하면, 공유메모리 버전의 전역 트랜잭션이 줄어든 것을 확인할 수 있습니다(08의 ncu 워크플로와 동일).

세 버전(naive → atomic-only → shared)을 각각 `ncu --set full`로 떠서 **atomic 관련 스톨(stall)
비중**을 비교해보면, naive는 애초에 정확성 문제로 의미 있는 비교 대상이 못 되고, atomic-only
버전은 "Global Atomic" 관련 대기 비중이 높게, shared 버전은 그 비중이 줄어드는 것을 수치로 확인할
수 있습니다.
</details>

## 🧪 추가 연습 & 비교

지금까지 만든 커널이 CuPy 내장 함수와 실제로 얼마나 비슷한 성능을 내는지, 그리고 처리량을 더
늘리는 방향으로 확장하면 어떻게 되는지 확인합니다.

**비교 — `cp.bincount` 대비**: CuPy 내장 히스토그램과 정확성·속도를 비교하세요.

`cp.bincount`는 CuPy가 내부적으로 이미 고도로 튜닝한 라이브러리 루틴입니다(정렬 기반이나 다단계
리덕션 등, 우리가 직접 짠 것보다 훨씬 정교한 전략을 쓸 수 있음). 여기서는 여러분이 직접 만든
`hist_shared`가 **정확성**(결과가 완전히 같은지)과 **속도**(라이브러리 대비 얼마나 뒤처지는지)
양쪽에서 어느 수준인지 가늠해보는 것이 목적입니다. 교육용으로 단계별 원리를 이해하기 위해 직접
커널을 짰지만, 실전에서는 이런 표준 문제일수록 **검증된 라이브러리 함수를 우선 고려**하는 것이
합리적입니다 — 이 관점은 `10_cccl`에서 "직접 커널 대신 검증된 병렬 알고리즘을 쓴다"는 주제로
이어집니다.

In [ ]:
def run_shared():
    h = cp.zeros(BINS, dtype=cp.int32); hist_shared[1024, threads](values, h); return h
h_cp = cp.bincount(values, minlength=BINS)
np.testing.assert_array_equal(cp.asnumpy(run_shared()), cp.asnumpy(h_cp)); print('shared == bincount')
print_bench(bench(lambda: cp.bincount(values, minlength=BINS), n_repeat=20, name='cp.bincount'))

**연습 — grid-stride + items_per_thread**: 각 스레드가 여러 값을 처리하도록 공유메모리 히스토그램을 확장하세요(블록 수↓, 처리량↑).
힌트: grid-stride 루프(`while i < n: ...; i += stride`)는 이미 적용돼 있습니다. 블록 수를 줄여(예: 256) 스레드당 처리량을 늘려 측정해 보세요.

### 조금 더 구체적으로

`hist_shared`는 이미 grid-stride 루프로 작성되어 있어 블록 수(`nblocks`)를 바꿔도 정확성에는
영향이 없습니다 — 단지 **블록 하나가 처리하는 값의 개수**(사실상 `items_per_thread`에 해당)만
달라집니다. 이는 4절에서 계산한 "전역 atomic 횟수 = `nblocks × BINS`" 공식과 바로 연결됩니다.
- `nblocks`를 **줄이면**: 최종 병합 단계의 전역 atomic 횟수(`nblocks × 256`)가 줄어 5절에서 본
  이득이 더 커지지만, 블록 하나가 처리해야 할 값이 늘어나 **블록 내부(공유메모리) 경합**은 오히려
  늘어납니다.
- `nblocks`를 **늘리면**: 반대로 블록 내부 경합은 줄지만 최종 병합의 전역 atomic 횟수가 늘어납니다.

즉 여기에는 07의 7절에서 다룬 **occupancy 트레이드오프**와 같은 성격의 최적점이 존재합니다 —
`08`에서 `threads_per_block`을 스윕해 최적점을 찾았듯, 아래 코드 셀은 `nblocks`를 스윕해 이
트레이드오프의 실제 모양을 측정합니다.

In [ ]:
for nblocks in [256, 1024, 4096]:
    def run(b=nblocks):
        h = cp.zeros(BINS, dtype=cp.int32); hist_shared[b, threads](values, h)
    print('blocks', nblocks, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

<a id="7"></a>
## 7. 체크포인트

이 노트북에서 다룬 3단계 — **레이스 관찰 → atomic으로 정확성 복구 → 공유메모리로 경합 감소** —
는 병렬 히스토그램뿐 아니라 리덕션·정렬·그래프 알고리즘 등 "여러 스레드가 같은 자원을 공유하는"
모든 병렬 패턴에 그대로 적용되는 사고 순서입니다.

- [ ] 병렬 누적의 데이터 레이스를 설명할 수 있다
- [ ] `cuda.atomic.add`로 안전하게 히스토그램을 만들었다
- [ ] 공유메모리 privatization + `syncthreads`로 가속했다
- [ ] 전역 vs 공유 atomic 성능을 비교했다

다음: **`10_cccl`** — 직접 커널을 짜는 대신 **검증된 병렬 알고리즘**(reduce/scan/transform)으로 같은 일을 합니다.